# Oiled — Stage 2: SAR Segmentation Baseline (Kaggle)
**2-Channel U-Net (ResNet-34) Trained on Part 3 SAR Dataset**  
Dataset: `02_Test_images_and_ground_truth.7z`  
Target: Sentinel-1 SAR Dual-Pol (VV + VH) $\rightarrow$ Binary Oil Spill Probability Mask

### Cell 1: GPU & Hardware Verification

In [ ]:
import torch
import platform

print("=== Oiled — Kaggle Hardware Check ===")
print(f"Python:  {platform.python_version()}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.version.cuda}")
print(f"GPUs:    {torch.cuda.device_count()}")

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected! In Kaggle sidebar, turn ON Accelerator -> GPU T4 x2 or P100.")

for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name} | VRAM: {p.total_memory // 1024**2} MiB")

DEVICE = torch.device("cuda:0")
N_GPU  = torch.cuda.device_count()

### Cell 2: Install Dependencies

In [ ]:
!apt-get install -y -qq p7zip-full
!pip install -q timm==0.9.7
!pip install -q segmentation-models-pytorch==0.3.4
!pip install -q rasterio==1.4.3
!pip install -q albumentations==1.4.23
!pip install -q scikit-learn
print("✓ Dependencies installed successfully.")

### Cell 3: Locate Archive & Extract to `/tmp` Scratch Disk
> **Kaggle Best Practice:** We extract to `/tmp/data/part3` instead of `/kaggle/working`. This prevents Kaggle from saving 10 GB of extracted raster files as output artifacts when the notebook finishes!

In [ ]:
import subprocess
from pathlib import Path

# Locate archive in current directory or /kaggle/input
archive_candidates = [
    Path("02_Test_images_and_ground_truth.7z"),
    Path("/kaggle/working/02_Test_images_and_ground_truth.7z"),
] + list(Path("/kaggle/input").rglob("02_Test_images_and_ground_truth.7z"))

ARCHIVE_PATH = next((p for p in archive_candidates if p.exists()), None)
if ARCHIVE_PATH is None:
    raise FileNotFoundError("Could not find 02_Test_images_and_ground_truth.7z in current directory or /kaggle/input!")

print(f"✓ Found archive: {ARCHIVE_PATH} ({ARCHIVE_PATH.stat().st_size / 1e6:.1f} MB)")

# Extract to /tmp scratch space
EXTRACT_DIR = Path("/tmp/data/part3")
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

existing = len(list(EXTRACT_DIR.rglob("*.tif")))
if existing == 900:
    print(f"✓ Data already extracted ({existing} GeoTIFFs). Skipping.")
else:
    print(f"Extracting {ARCHIVE_PATH.name} to {EXTRACT_DIR}...")
    subprocess.run(["7z", "x", str(ARCHIVE_PATH), f"-o{EXTRACT_DIR}", "-y"], check=True)
    total_tifs = len(list(EXTRACT_DIR.rglob("*.tif")))
    print(f"✓ Extraction complete. Total GeoTIFFs: {total_tifs}")

### Cell 4: Exact Image-Mask Pairing & Category Split

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

images_root = EXTRACT_DIR / "Images"
masks_root  = EXTRACT_DIR / "Mask"

def get_category(path: Path) -> str:
    s = str(path).lower()
    if "lookalike" in s:
        return "lookalike"
    elif "no" in s:
        return "clean"
    else:
        return "oil"

# Index masks by (category, clean_id)
mask_index = {}
for m in masks_root.rglob("*.tif"):
    cat = get_category(m)
    clean_id = m.stem.replace("_segmentation", "").replace("_mask", "").replace("mask", "").strip("_")
    mask_index[(cat, clean_id)] = m

# Pair with images
paired = []
for img in sorted(images_root.rglob("*.tif")):
    cat = get_category(img)
    clean_id = img.stem.replace("_image", "").replace("image", "").strip("_")
    match = mask_index.get((cat, clean_id))
    if match and match.exists():
        paired.append({
            "scene_id": f"{cat}_{clean_id}",
            "category": cat,
            "image": str(img),
            "mask": str(match)
        })

manifest_df = pd.DataFrame(paired)
print(f"✓ Successfully paired scenes: {len(manifest_df)} / 450")
print("Category distribution:")
print(manifest_df["category"].value_counts().to_string())

# Filter to Oil scenes (150 scenes) and perform 85% train / 15% validation split
oil_df = manifest_df[manifest_df["category"] == "oil"].reset_index(drop=True)
train_df, val_df = train_test_split(oil_df, test_size=0.15, random_state=42, shuffle=True)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)

print(f"\n✓ Train scenes: {len(train_df)} | Val scenes: {len(val_df)}")

# Save manifest to working directory
OUT_DIR = Path("/kaggle/working")
manifest_df.to_csv(OUT_DIR / "manifest_part3_paired.csv", index=False)
train_df.to_csv(OUT_DIR / "train_scenes.csv", index=False)
val_df.to_csv(OUT_DIR / "val_scenes.csv", index=False)

### Cell 5: 512×512 Tile Extraction & DataLoaders

In [ ]:
import numpy as np
import rasterio
from rasterio.errors import NotGeoreferencedWarning
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings("ignore", category=NotGeoreferencedWarning)

SAR_MIN, SAR_MAX = -40.0, 10.0
TILE_SIZE = 512
TILE_STRIDE = 384
MAX_TILES_PER_SCENE = 25

def read_sar(path: str) -> np.ndarray:
    with rasterio.open(path) as src:
        arr = src.read().astype(np.float32)
    if arr.shape[0] == 1:
        arr = np.concatenate([arr, arr], axis=0)
    arr = arr[:2]  # VV and VH bands
    arr = np.clip(arr, SAR_MIN, SAR_MAX)
    arr = (arr - SAR_MIN) / (SAR_MAX - SAR_MIN)
    return np.transpose(arr, (1, 2, 0))

def read_mask(path: str) -> np.ndarray:
    with rasterio.open(path) as src:
        m = src.read(1).astype(np.uint8)
    return np.clip(m, 0, 1)

def extract_tiles(img, mask, tile_size=TILE_SIZE, stride=TILE_STRIDE):
    H, W = img.shape[:2]
    for y in range(0, max(H - tile_size + 1, 1), stride):
        for x in range(0, max(W - tile_size + 1, 1), stride):
            it = img[y:y+tile_size, x:x+tile_size]
            mt = mask[y:y+tile_size, x:x+tile_size] if mask is not None else np.zeros((tile_size, tile_size), np.uint8)
            if it.shape[:2] == (tile_size, tile_size):
                yield it, mt

AUG_TRAIN = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.GaussNoise(var_limit=(0.002, 0.015), p=0.3),
    ToTensorV2()
])
AUG_VAL = A.Compose([ToTensorV2()])

class SARSpiDataset(Dataset):
    def __init__(self, df, augment=True):
        self.aug = AUG_TRAIN if augment else AUG_VAL
        self.tiles = []
        tag = "train" if augment else "val"
        print(f"Pre-caching 512x512 tiles for {tag} ({len(df)} scenes)...")
        for _, row in tqdm(df.iterrows(), total=len(df), desc=tag):
            try:
                img = read_sar(row["image"])
                mask = read_mask(row["mask"])
                count = 0
                for it, mt in extract_tiles(img, mask):
                    self.tiles.append((it.copy(), mt.copy()))
                    count += 1
                    if count >= MAX_TILES_PER_SCENE:
                        break
            except Exception as e:
                pass
        print(f"  [{tag}] Loaded {len(self.tiles)} tiles.")

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, idx):
        img, mask = self.tiles[idx]
        out = self.aug(image=img, mask=mask)
        return out["image"].float(), out["mask"].float().unsqueeze(0)

train_ds = SARSpiDataset(train_df, augment=True)
val_ds   = SARSpiDataset(val_df,   augment=False)

BATCH_SIZE = 16 * N_GPU
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
print(f"\n✓ DataLoaders ready: {len(train_loader)} train batches | {len(val_loader)} val batches (Batch size={BATCH_SIZE})")

### Cell 6: Model Architecture & BCEDiceLoss

In [ ]:
import torch.nn as nn
import segmentation_models_pytorch as smp

# 2-channel U-Net with ResNet-34 encoder
model = smp.Unet(
    encoder_name="resnet34",
    encoder_weights=None,
    in_channels=2,
    classes=1,
    activation=None
)

if N_GPU > 1:
    model = nn.DataParallel(model)
    print(f"✓ Enabled DataParallel across {N_GPU} GPUs.")

model = model.to(DEVICE)

class BCEDiceLoss(nn.Module):
    def __init__(self, w=0.5, smooth=1.0):
        super().__init__()
        self.w = w
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
    def dice(self, logits, target):
        p = torch.sigmoid(logits)
        num = 2.0 * (p * target).sum(dim=(2, 3)) + self.smooth
        den = p.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + self.smooth
        return 1.0 - (num / den).mean()
    def forward(self, logits, target):
        return self.w * self.bce(logits, target) + (1.0 - self.w) * self.dice(logits, target)

def calc_iou(logits, target, thresh=0.5, eps=1e-6):
    p = (torch.sigmoid(logits) > thresh).float()
    inter = (p * target).sum()
    union = p.sum() + target.sum() - inter
    return ((inter + eps) / (union + eps)).item()

def calc_dice(logits, target, thresh=0.5, eps=1e-6):
    p = (torch.sigmoid(logits) > thresh).float()
    return ((2.0 * (p * target).sum() + eps) / (p.sum() + target.sum() + eps)).item()

criterion = BCEDiceLoss(w=0.5)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)
scaler    = torch.cuda.amp.GradScaler()

print(f"✓ Model ready ({sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M params).")

### Cell 7: Fast GPU Mixed-Precision Training

In [ ]:
import time

EPOCHS = 20
best_val_iou = 0.0
history = []

print(f"=== Starting GPU Training on {N_GPU}x {torch.cuda.get_device_name(0)} ===")
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    # Train
    model.train()
    tr_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        tr_loss += loss.item()
    tr_loss /= len(train_loader)

    # Validation
    model.eval()
    va_loss = va_iou = va_dice = 0.0
    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            with torch.cuda.amp.autocast():
                logits = model(xb)
                va_loss += criterion(logits, yb).item()
            va_iou  += calc_iou(logits, yb)
            va_dice += calc_dice(logits, yb)
            
    va_loss /= len(val_loader)
    va_iou  /= len(val_loader)
    va_dice /= len(val_loader)
    scheduler.step()

    elapsed = time.time() - t0
    history.append({"epoch": epoch, "train_loss": tr_loss, "val_loss": va_loss, "val_iou": va_iou, "val_dice": va_dice})
    print(f"Epoch {epoch:2d}/{EPOCHS} | Tr Loss: {tr_loss:.4f} | Val Loss: {va_loss:.4f} | Val IoU: {va_iou:.4f} | Val Dice: {va_dice:.4f} | {elapsed:.0f}s")

    # Checkpoint on best validation IoU
    if va_iou > best_val_iou:
        best_val_iou = va_iou
        state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
        torch.save({
            "epoch": epoch,
            "model_state": state,
            "val_iou": best_val_iou,
            "val_dice": va_dice,
        }, OUT_DIR / "seg_best.pt")
        print(f"  ✓ Best model saved (Val IoU: {best_val_iou:.4f})")

print(f"\n✓ Training complete! Best Validation IoU: {best_val_iou:.4f}")

### Cell 8: ONNX Export & Metrics Summary

In [ ]:
import json
from datetime import datetime

# Export to ONNX
export_model = smp.Unet("resnet34", encoder_weights=None, in_channels=2, classes=1, activation="sigmoid").to(DEVICE).eval()
ckpt = torch.load(OUT_DIR / "seg_best.pt", map_location=DEVICE)
export_model.load_state_dict(ckpt["model_state"])

dummy = torch.randn(1, 2, 512, 512, device=DEVICE)
onnx_path = OUT_DIR / "unet_segmentation.onnx"

torch.onnx.export(
    export_model, dummy, str(onnx_path),
    input_names=["sar_tile"], output_names=["oil_probability"],
    dynamic_axes={"sar_tile": {0: "batch"}, "oil_probability": {0: "batch"}},
    opset_version=17
)
print(f"✓ Exported ONNX: {onnx_path} ({onnx_path.stat().st_size / 1e6:.1f} MB)")

# Save training summary
metrics = {
    "completed_utc": datetime.utcnow().isoformat(),
    "hardware": f"{N_GPU}x {torch.cuda.get_device_name(0)}",
    "best_val_iou": round(best_val_iou, 4),
    "history": history
}
with open(OUT_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\n=== Saved Outputs in /kaggle/working ===")
for f in sorted(OUT_DIR.glob("*")):
    if f.is_file():
        print(f"  {f.name:30s} {f.stat().st_size / 1e6:6.2f} MB")